Imports

In [1]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

Initial exploration

In [ ]:
data_dir = "../data/raw"
xml_files = [f for f in os.listdir(data_dir) if f.endswith(".xml")]
xml_files.sort()

print(f"Number of XML files: {len(xml_files)}")
print("\nStructure of a sample file (Ahmad al-Mansur):")
tree = ET.parse(os.path.join(data_dir, "CIV5Leader_AhmadalMansur.xml"))
root = tree.getroot()
print(f"Available sections: {[child.tag for child in root]}")
print(f"Number of flavors: {len(root.findall('.//Leader_Flavors/Row'))}")
print(f"Number of major civ approaches: {len(root.findall('.//Leader_MajorCivApproachBiases/Row'))}")
print(f"Number of minor civ approaches: {len(root.findall('.//Leader_MinorCivApproachBiases/Row'))}")

Parsing function

In [ ]:
def parse_all_leaders(data_dir, xml_files):
    EXCLUDES = {
        "CIV5LeaderTables.xml",
        "CIV5Leaders_Inherited_Expansion2.xml",
        "CIV5Leader_Barbarian.xml"
    }
    DIPLO_FIELDS = [
        "VictoryCompetitiveness", "WonderCompetitiveness", "MinorCivCompetitiveness",
        "Boldness", "DiploBalance", "WarmongerHate", "DenounceWillingness",
        "DoFWillingness", "Loyalty", "Neediness", "Forgiveness", "Chattiness", "Meanness"
    ]
    leader_files = {}
    for f in xml_files:
        if f in EXCLUDES:
            continue
        name = f.replace(".xml", "").replace("CIV5Leader_", "")
        if "Expansion2" in name:
            base = name.replace("_Expansion2", "")
            leader_files.setdefault(base, {})["expansion2"] = f
        elif "Expansion" in name:
            base = name.replace("_Expansion", "")
            leader_files.setdefault(base, {})["expansion"] = f
        else:
            leader_files.setdefault(name, {})["vanilla"] = f

    def has_leader_block(filepath):
        root = ET.parse(os.path.join(data_dir, filepath)).getroot()
        return root.find(".//Leaders/Row") is not None

    def find_base_file(files_dict):
        for key in ["vanilla", "expansion", "expansion2"]:
            if key in files_dict and has_leader_block(files_dict[key]):
                return files_dict[key]
        return None

    all_leaders = []
    for base_name, files_dict in leader_files.items():
        base_file = find_base_file(files_dict)
        if base_file is None:
            print(f"Warning: no base data found for {base_name}")
            continue
        root = ET.parse(os.path.join(data_dir, base_file)).getroot()
        leader_row = root.find(".//Leaders/Row")
        leader = {"leader_name": base_name, "leader_type": leader_row.findtext("Type")}
        for field in DIPLO_FIELDS:
            val = leader_row.findtext(field)
            leader[field] = int(val) if val is not None else None
        for row in root.findall(".//Leader_MajorCivApproachBiases/Row"):
            approach = row.findtext("MajorCivApproachType").replace("MAJOR_CIV_APPROACH_", "MAJOR_")
            leader[approach] = int(row.findtext("Bias"))
        for row in root.findall(".//Leader_MinorCivApproachBiases/Row"):
            approach = row.findtext("MinorCivApproachType").replace("MINOR_CIV_APPROACH_", "MINOR_")
            leader[approach] = int(row.findtext("Bias"))
        for row in root.findall(".//Leader_Flavors/Row"):
            flavor = row.findtext("FlavorType").replace("FLAVOR_", "")
            leader[flavor] = int(row.findtext("Flavor"))
        for key in ["expansion", "expansion2"]:
            if key in files_dict and files_dict[key] != base_file:
                root2 = ET.parse(os.path.join(data_dir, files_dict[key])).getroot()
                for row in root2.findall(".//Leader_Flavors/Row"):
                    flavor = row.findtext("FlavorType").replace("FLAVOR_", "")
                    leader[flavor] = int(row.findtext("Flavor"))
        all_leaders.append(leader)
    return pd.DataFrame(all_leaders)

Run, clean and save

In [ ]:
df = parse_all_leaders(data_dir, xml_files)
cols_to_fill = ["MINOR_BULLY", "USE_NUKE", "ESPIONAGE", "ANTIAIR", "AIR_CARRIER"]
df[cols_to_fill] = df[cols_to_fill].fillna(5)
print(f"Final dataset: {df.shape[0]} leaders x {df.shape[1]} variables")
df.to_csv("../data/civ5_leaders.csv", index=False)
print("Dataset saved!")